In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
from src.soc_engine import (
    BatterySpec, DispatchAction, SoCEngine,
    simulate_dispatch_schedule, calculate_revenue, soc_diagnostics
)

passed = 0
failed = 0

def check(condition, msg_pass, msg_fail):
    global passed, failed
    if condition:
        print(f"✅ {msg_pass}")
        passed += 1
    else:
        print(f"❌ {msg_fail}")
        failed += 1

# Standard 100MW / 4h spec for all tests
spec = BatterySpec(
    power_mw=100,
    duration_hours=4,
    charge_eff=0.92,
    discharge_eff=0.92,
    min_soc_pct=10.0,
    max_soc_pct=100.0,
    initial_soc_pct=50.0,
    ramp_rate_pct=100.0,
)

# ── Test 1 — BatterySpec properties ───────────────────────────────────────
check(spec.energy_capacity_mwh == 400,  "Test 1 — energy capacity = 400 MWh", "Test 1 — wrong capacity")
check(spec.min_soc_mwh         ==  40,  "Test 1 — min SoC = 40 MWh",         "Test 1 — wrong min SoC")
check(spec.max_soc_mwh         == 400,  "Test 1 — max SoC = 400 MWh",        "Test 1 — wrong max SoC")
check(spec.initial_soc_mwh     == 200,  "Test 1 — initial SoC = 200 MWh",    "Test 1 — wrong initial SoC")
check(abs(spec.round_trip_efficiency - 0.92 * 0.92) < 1e-6,
      f"Test 1 — RTE = {spec.round_trip_efficiency:.4f}",
      "Test 1 — wrong RTE")

# ── Test 2 — SoC initialises correctly ────────────────────────────────────
engine = SoCEngine(spec)
check(engine.soc_mwh == 200, "Test 2 — SoC initialises at 200 MWh", f"Test 2 — SoC = {engine.soc_mwh}")
check(engine.soc_pct == 50,  "Test 2 — SoC % initialises at 50%",   f"Test 2 — SoC% = {engine.soc_pct}")

# ── Test 3 — No simultaneous charge/discharge ──────────────────────────────
try:
    bad_action = DispatchAction(charge_mw=50, discharge_mw=50)
    print("❌ Test 3 — should have raised AssertionError for simultaneous charge/discharge")
    failed += 1
except AssertionError:
    check(True, "Test 3 — simultaneous charge/discharge correctly rejected", "")

# ── Test 4 — Charge increases SoC correctly ───────────────────────────────
engine.reset()
soc_before = engine.soc_mwh
result = engine.step(DispatchAction(charge_mw=100))
expected_energy = 100 * 0.92   # 92 MWh added to battery
check(
    abs(result["energy_charged_mwh"] - expected_energy) < 0.01,
    f"Test 4 — charging 100MW adds {expected_energy} MWh to battery",
    f"Test 4 — energy charged = {result['energy_charged_mwh']}, expected {expected_energy}"
)
check(
    abs(engine.soc_mwh - (soc_before + expected_energy)) < 0.01,
    f"Test 4 — SoC increases by {expected_energy} MWh after charging",
    f"Test 4 — SoC = {engine.soc_mwh}, expected {soc_before + expected_energy}"
)

# ── Test 5 — Discharge delivers less than removed (efficiency loss) ────────
engine.reset()
result = engine.step(DispatchAction(discharge_mw=100))
energy_removed    = result["energy_discharged_mwh"]   # from battery
energy_delivered  = result["energy_delivered_mwh"]    # to grid
check(
    abs(energy_delivered - energy_removed * 0.92) < 0.01,
    f"Test 5 — energy delivered = energy removed × discharge_eff: "
    f"{energy_delivered:.2f} = {energy_removed:.2f} × 0.92",
    "Test 5 — efficiency not applied correctly on discharge"
)

# ── Test 6 — Cannot discharge below min SoC ───────────────────────────────
engine.reset()
engine.soc_mwh = spec.min_soc_mwh + 10   # just above minimum
result = engine.step(DispatchAction(discharge_mw=100))
check(
    engine.soc_mwh >= spec.min_soc_mwh,
    f"Test 6 — SoC never drops below min ({spec.min_soc_pct}%): {engine.soc_pct:.1f}%",
    f"Test 6 — SoC dropped below minimum: {engine.soc_pct:.1f}%"
)

# ── Test 7 — Cannot charge above max SoC ──────────────────────────────────
engine.reset()
engine.soc_mwh = spec.max_soc_mwh - 5    # just below maximum
result = engine.step(DispatchAction(charge_mw=100))
check(
    engine.soc_mwh <= spec.max_soc_mwh,
    f"Test 7 — SoC never exceeds max ({spec.max_soc_pct}%): {engine.soc_pct:.1f}%",
    f"Test 7 — SoC exceeded maximum: {engine.soc_pct:.1f}%"
)

# ── Test 8 — Ramp rate limits enforced ────────────────────────────────────
ramp_spec = BatterySpec(
    power_mw=100, duration_hours=4,
    ramp_rate_pct=50.0   # max 50 MW per hour ramp
)
ramp_engine = SoCEngine(ramp_spec)
result = ramp_engine.step(DispatchAction(charge_mw=100))
check(
    result["charge_mw_actual"] <= 50.0,
    f"Test 8 — ramp rate limits charge to 50 MW: {result['charge_mw_actual']} MW",
    f"Test 8 — ramp rate not enforced: {result['charge_mw_actual']} MW"
)

# ── Test 9 — Reset restores initial state ─────────────────────────────────
engine.reset()
engine.step(DispatchAction(charge_mw=100))
engine.step(DispatchAction(discharge_mw=50))
engine.reset()
check(
    abs(engine.soc_mwh - spec.initial_soc_mwh) < 0.01,
    f"Test 9 — reset restores SoC to {spec.initial_soc_mwh} MWh",
    f"Test 9 — reset failed: SoC = {engine.soc_mwh}"
)

# ── Test 10 — simulate_dispatch_schedule returns correct shape ─────────────
n_hours = 48
schedule = pd.DataFrame({
    "charge_mw":    [100 if h % 24 < 8 else 0  for h in range(n_hours)],
    "discharge_mw": [100 if h % 24 >= 16 else 0 for h in range(n_hours)],
}, index=pd.date_range("2024-01-01", periods=n_hours, freq="h"))

dispatch_df = simulate_dispatch_schedule(spec, schedule)
check(len(dispatch_df) == n_hours,
      f"Test 10 — dispatch returns {n_hours} rows",
      f"Test 10 — wrong row count: {len(dispatch_df)}")
check(
    all(c in dispatch_df.columns for c in [
        "energy_charged_mwh", "energy_delivered_mwh",
        "soc_pct_after", "charge_clipped"
    ]),
    "Test 10 — all expected columns present",
    "Test 10 — missing columns in dispatch output"
)

# ── Test 11 — SoC stays within bounds throughout full simulation ───────────
check(
    dispatch_df["soc_pct_after"].min() >= spec.min_soc_pct - 0.01,
    f"Test 11 — SoC never below min throughout simulation: "
    f"min={dispatch_df['soc_pct_after'].min():.2f}%",
    f"Test 11 — SoC violated min bound: {dispatch_df['soc_pct_after'].min():.2f}%"
)
check(
    dispatch_df["soc_pct_after"].max() <= spec.max_soc_pct + 0.01,
    f"Test 11 — SoC never above max throughout simulation: "
    f"max={dispatch_df['soc_pct_after'].max():.2f}%",
    f"Test 11 — SoC violated max bound: {dispatch_df['soc_pct_after'].max():.2f}%"
)

# ── Test 12 — calculate_revenue computes correct net revenue ───────────────
prices = pd.Series(
    [20.0] * 8 + [50.0] * 8 + [20.0] * 8 + [50.0] * 8,
    index=schedule.index
)
revenue_df = calculate_revenue(dispatch_df, prices)
check(
    "net_revenue" in revenue_df.columns,
    "Test 12 — net_revenue column present",
    "Test 12 — net_revenue column missing"
)
check(
    "cumulative_revenue" in revenue_df.columns,
    "Test 12 — cumulative_revenue column present",
    "Test 12 — cumulative_revenue column missing"
)
# Charging at $20, delivering at $50 × discharge_eff — should be profitable
total_rev = revenue_df["net_revenue"].sum()
check(
    total_rev > 0,
    f"Test 12 — total revenue is positive: ${total_rev:,.2f}",
    f"Test 12 — revenue should be positive: ${total_rev:,.2f}"
)

# ── Test 13 — soc_diagnostics returns correct structure ───────────────────
diag = soc_diagnostics(dispatch_df, spec)
required_keys = [
    "min_soc_pct", "max_soc_pct", "avg_soc_pct",
    "charge_clipped_hours", "discharge_clipped_hours",
    "total_energy_charged_mwh", "total_energy_delivered_mwh",
    "effective_rte", "total_cycles"
]
for key in required_keys:
    check(key in diag,
          f"Test 13 — diagnostic key '{key}' present",
          f"Test 13 — diagnostic key '{key}' MISSING")

check(
    diag["effective_rte"] <= 1.0,
    f"Test 13 — effective RTE <= 1.0: {diag['effective_rte']:.4f}",
    "Test 13 — effective RTE > 1.0 — energy created from nothing"
)

# ── Test 14 — zero Streamlit imports ──────────────────────────────────────
with open("../src/soc_engine.py") as f:
    source = f.read()
check(
    "import streamlit" not in source and "from streamlit" not in source,
    "Test 14 — src/soc_engine.py has no Streamlit imports",
    "Test 14 — Streamlit imports found in soc_engine.py"
)

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n{'─' * 50}")
print(f"Results: {passed} passed / {failed} failed / {passed + failed} total")
if failed == 0:
    print("🎉 All tests passed — SoC engine ready for Step 3")
else:
    print("⚠️  Fix failing tests before proceeding to forecast integration")
print(f"{'─' * 50}")